# Introduction to Convolutional Neural Network and Computer Vision with TensorFlow

Computer Vision is the practice of writing alogorithm which can discover patterns in visua data. Such as the camera of self-driving car recognizing the car in front.

### Convolutional Neural Network (CNN)

> **Architecture**

* input_image - Target image for which we want to discover the pattern
* Input layer - take the image and preprocess it for further layers
>           `input_shapes = [batch_size, image_height, width_height, color_channels]`
* Convolutional layer - Extract/learns the most important features from target images
>            `Multiple, can create with tf.keras.layers.ConvXD` (X can be multiple values)
* Hidden activation - Add non-linearity to learned features (non-staright lines)
>            `Usually ReLU`
* Pooling layer - Reduces the dimensionality of learned image features
>            Average or Max `tf.keras.layers.AvgPool2D/MaxPool2D`
* Fully connected layer - Further refines learned features from convolutional layers
* Output layer - Takes learned features and outputs them in shape of target labels
>            `output_shape = [number_of_classes]`
* Output activation - Adds non-linearity to output layer
>               `sigmoid for Binary and softmax for Multiclass`



### Get the data

The image we're working with are from the Food101 dataset (101 different classes of food): https://www.kaggle.com/datasets/dansbecker/food-101

However we'vw modified it to only use two classes (pizzas 🍕 & steak 🥩) using the image  data modification notebbok: https://github.com/mrdbourke/tensorflow-deep-learning/blob/main/extras/image_data_modification.ipynb

> 🔑 **Note**: We start with a smaller dataset so we can experiment quickly and figure what works (or better yet what doesn't work) before scaling up.

In [ ]:
import zipfile

!curl -O https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip

In [ ]:
# Unzip the downloaded file
zip_ref = zipfile.ZipFile('../../data/pizza_steak.zip')
zip_ref.extractall()
zip_ref.close()

### Inspect the data (become one with data)

A very crucial step at the begining of any machine learninf project is becoming one with data.

And for a computer vision project... this usually means visualizing many sampkles of the data.

In [ ]:
!ls  ../../data/pizza_steak/train/

In [ ]:
!ls  ../../data/pizza_steak/train/steak

In [ ]:
!ls  ../../data/pizza_steak/train/steak | wc -l

In [ ]:
import os 

# Walk through pizza_steak directory which is in data directory and list number of files
for dirpath, dirnames, filenames in os.walk('../../data/pizza_steak'):
    print(f'There are {len(dirnames)} directories and {len(filenames)} images in {dirpath}.')

In [ ]:
!ls -la ../../data/pizza_steak/

In [ ]:
# another way to find how many images are in a file
num_images_steak_train = len(os.listdir('../../data/pizza_steak/train/steak'))
num_images_steak_train

To visulaise our imahges, let's first get the class name programmatically

In [ ]:
# get the classname programmatically
import pathlib
import numpy as np

data_dir = pathlib.Path('../../data/pizza_steak/train')
class_names = np.array(sorted([item.name for item in data_dir.glob('*')])) # Create a list of class_namesfrom the subdirectories

print(class_names)

In [ ]:
# Lets visualize our images
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import random

def view_random_image(target_dir, target_class):
    '''Setup the target directories (we'll view the images from here)'''
    target_folder = target_dir+target_class
    
    # get a random image path
    random_image = random.sample(os.listdir(target_folder), 1)
    print(random_image)
    
    # read in the image and plot it 
    img = mpimg.imread(target_folder + "/" + random_image[0])
    plt.imshow(img)
    plt.title(target_class)
    plt.axis("off")
    
    print(f"Image shape: {img.shape}") # shwo the shape of the image
    
    return img

    

In [ ]:
# view a random image from the training dataset
img = view_random_image(target_dir='../../data/pizza_steak/train/',
                        target_class="steak")

In [ ]:
import tensorflow as tf
tf.constant(img)

In [ ]:
img.shape  # width, heigh, color_channels

🔑 **Note**: s we've discussed before, many machine learning models icluding neural network prefer the values they work with to be between 0 and 1. Knowing this, one of the most common preprocessing steps for working with images is to **scale** (also referred to as **normalize**) their pixel values by dividing he image arrays by 255. (since 255 is the maximum pixel value).

In [ ]:
# get all the pixel values between 0 and 1 
img / 255.0

### An end-to-end example

Let's build a convolutional neural network to find patterns in our images, more specifically we a need way to:

* Load our images
* Preprocess our images
* Build a CNN to find patterns in our images
* Compile our CNN
* Fit the CNN to our training data

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# set the seed
tf.random.set_seed(42)

# preprocess our data (scaling/normalization)
train_datagen = ImageDataGenerator(rescale=1/255.)
valid_datagen = ImageDataGenerator(rescale=1/255.)

# Setup paths to our data directories
train_dir = '../../data/pizza_steak/train'
test_dir = '../../data/pizza_steak/test'

# Import datafrom directories and turn it into batches
train_data = train_datagen.flow_from_directory(directory=train_dir,
                                               batch_size=32,
                                               target_size=(224, 224),
                                               class_mode='binary',
                                               seed=42)

valid_data = valid_datagen.flow_from_directory(directory=test_dir,
                                               batch_size=32,
                                               target_size=(224, 224),
                                               class_mode='binary',
                                               seed=42)

# Build a CNN
model_1 = tf.keras.Sequential([
    tf.keras.layers.Conv2D(filters=10,
                                 kernel_size=3,
                                 activation='relu',
                                 input_shape=(224, 224, 3)),
    
    tf.keras.layers.Conv2D(10, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(pool_size=2,
                              padding='valid'),
    tf.keras.layers.Conv2D(10, 3, activation='relu'),
    tf.keras.layers.Conv2D(10, 3, activation='relu'),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation='sigmoid')   
])

# compile our CNN
model_1.compile(loss='binary_crossentropy',
                optimizer='adam',
                metrics=['accuracy'])

# fit the model
history_1 = model_1.fit(train_data,
                        epochs=5,
                        steps_per_epoch=len(train_data),
                        validation_data=valid_data,
                        validation_steps=len(valid_data))

In [ ]:
# Get model 1 summary
model_1.summary()

📖 https://poloclub.github.io/cnn-explainer/

One of the best resources on `CNN`

### Using the same model as before

Let's replicate the model we built in previous section to see if it works with our image data.

The model we're building is from [TensorFlow playground](https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=2,2,2,2&seed=0.47032&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false)



In [ ]:
# set random seed
tf.random.set_seed(42)

# create a model to replicate the TensorFlow Playground model
model_2 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(224, 224, 3)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

# compile the model
model_2.compile(loss='binary_crossentropy',
                optimizer=tf.keras.optimizers.Adam(),
                metrics=['accuracy'])

# fit the model
history_2 = model_2.fit(train_data,
                        epochs=5,
                        steps_per_epoch=len(train_data),
                        validation_data=valid_data,
                        validation_steps=len(valid_data))

In [ ]:
# get the summary of model 2
model_2.summary()

Despite having 20x more parameter than our CNN (model_1), model_1 performs terribly, lets try to improve it

In [ ]:
# set random seed
tf.random.set_seed(42)

# Create the model 3
model_3 = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(224, 224, 3)),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# compile the model
model_3.compile(loss='binary_crossentropy',
                optimizer=tf.keras.optimizers.Adam(),
                metrics=['accuracy'])

# fit the model
history_3 = model_3.fit(train_data,
                        epochs=5,
                        steps_per_epoch=len(train_data),
                        validation_data=valid_data,
                        validation_steps=len(valid_data))

In [ ]:
# get the summary of model 3
model_3.summary()

Our CNN has 500x less parameters, so larger *trainable paraeters* are not always good, and we can safely draw conclusion that **Convolutional Neural Networks seek to sort out and learn the most important patterns in an image**. So even though these are less trainable paramters in our convolutional neural network, these are more helpful in deciphering between different *features* in an image.

In [ ]:
model_1.summary()

### Binary Classification: Let's break it down:

1. Become one with the data (visualize, visualize, visualize)
2. Preprocess the data (prepare it for our model, the main step here is scaling/normalizing) and turning our data into batches
3. Create a model (Baseline model)
4. Fit the model
5. Evaluate the model
6. Adjust different parameters and improve the model (try to beat our baseline)
7. Repeat until satisfies (Experiment, Experiment, Experiment)

### 1. Become one with the data

In [ ]:
# visualize
plt.figure()
plt.subplot(1, 2, 1)
steak_img = view_random_image(target_dir='../../data/pizza_steak/train/',
                              target_class="steak")
plt.subplot(1, 2, 2)
steak_img = view_random_image(target_dir='../../data/pizza_steak/train/',
                              target_class="pizza")

### 2. Preprocess the data (prepare it for our model)

In [ ]:
# define directory datset path
train_dir = '../../data/pizza_steak/train'
test_dir = '../../data/pizza_steak/test'

Our next step is to turn our data into **batches**.

A batch is a small subset of data, rather than look at all -10000 images at one time, a model might only look at 32 at a time.

It does this for couple of reasons:
1. 10,000 images (or more) might not fit into memory of your processor (GPU).
2. Trying to learn the pattern in 10k images in one hit could result in the same model not being able to learn very well.

Why 32?

Because 32 is good for your Health...
> * `Yann LeCun` - founder of Convolutional Neural Network

In [ ]:
# create train and test data generator  and rescale the data
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

In [ ]:
# loas im our image data from directories abd turn then into batches
train_data = train_datagen.flow_from_directory(directory=train_dir,   # target directory of iages
                                               target_size=(224, 224), # target size of images (h, w)
                                               class_mode='binary', # type of data working with
                                               batch_size=32) # size of minibatches to load data into

test_data = test_datagen.flow_from_directory(directory=test_dir,
                                               target_size=(224, 224),
                                               class_mode='binary',
                                               batch_size=32)

In [ ]:
# get a sampleo f training dataset
images, labels = next(train_data) # get tge 'next'batches of images/labels in train_data
len(images), len(labels)

In [ ]:
# how many batches are there
len(train_data)

In [ ]:
# get the first two images
images[:2], images[0].shape

In [ ]:
# view the first batch of labels
labels

### 3. Create a CNN model (start with Baseline)

A baseline model is a relatively simple model or existing result that you setup when begining a machine learning experminent  and the nas you keep experimenting you try to beat baseline.

> **Note** In deep learning there is almost an infinite amount of acrchitecture you could create. So one of the best wats to get started is to start with something simple and see, if it works on the data and then introduce comolexity as required.

In [ ]:
# AMke a creating ofour model little easier
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D, Activation
from tensorflow.keras import Sequential

In [ ]:
# create the model (this will be our baseline, a layer convolutional neural network)
model_4 = Sequential([
    Conv2D(filters=10, # number of sliding window going across an input, higher is more complex
           kernel_size=(3, 3), # size of sliding window going across an input
           strides=(1, 1), # size of step sliding window take across
           padding='same',  # if "same" output shape will be same as input shape, if "valid" output shape gets compressed
           activation='relu',
           input_shape=(224, 224, 3)),  # input layer (specify input shape)
    Conv2D(10 , 3, activation='relu'),
    Conv2D(10 , 3, activation='relu'),
    Flatten(),
    Dense(1, activation='sigmoid')  # output layer (working with binary classification so only one output)
])

In [ ]:
# compile the model
model_4.compile(loss='binary_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

In [ ]:
# mdoel summary
model_4.summary()

### 4. Fit the model

In [ ]:
# check the length of train and test data generator
len(train_data), len(test_data)

In [ ]:
# Fit the model
history_4 = model_4.fit(train_data,
                        epochs=5,
                        steps_per_epoch=len(train_data),
                        validation_data=test_data,
                        validation_steps=len(test_data))

### 5. Evaluate our model

It looks like our model is learning something, let's evaluate it.

In [ ]:
import pandas as pd
pd.DataFrame(history_4.history).plot(title='Model 4 evaluation')

In [ ]:
# plot validation and training curves seperately
def plot_loss_curves(history):
    ''' 
    Returns seperate loss curves for traning and validation metrics.
    '''
    loss = history.history['loss']
    val_loss = history.history["val_loss"]
    
    accuracy = history.history['accuracy']
    val_accuracy = history.history["val_accuracy"]
    
    epochs = range(len(history.history["loss"])) # how many epochs did we run for
    
    # plot loss
    plt.plot(epochs, loss, label="training loss")
    plt.plot(epochs, val_loss, label="val_loss")
    plt.xlabel("epochs")
    plt.title("loss")
    plt.legend()
    
    # plot accuracy
    plt.figure()
    plt.plot(epochs, accuracy, label="training accuracy")
    plt.plot(epochs, val_accuracy, label="val_accuracy")
    plt.xlabel("epochs")
    plt.title("accuracy")
    plt.legend()

> 🔑 **Note**: When a model's **validation loss starts to increase** it's likely that the model is **overfitting** the training dataset. This means, it's learnkng the pattern in training dataset *to well* and thus th model's ability to generalize to unseen data will be diminished.

In [ ]:
# loss and accuracy of model 4
plot_loss_curves(history_4)

### 6. Adjust the model parameter

Fitting a machine learning model

0. Create a baseline
1. Beat the baseline by overfitting a larger model
2. Reduce overfitting

> Ways to induce overfitting:
 * Increase the number of conv layers
 * Increase the number of conv filters
 * add another dense layer to the output of our Flatttend layer

> Reduce Overfitting:
 * Add data augumentation
 * Add regularization layers (such as MaxPool2D)
 * Add more data...

 > 🔑 **Note**: reducing overfitting is also called as *regularization*.

In [ ]:
# create a new model (this will be our new baseline)
model_5 = Sequential([
    Conv2D(10, 3, activation='relu', input_shape=(224, 224, 3)),
    MaxPool2D(pool_size=3),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(1, activation='sigmoid')
])

In [ ]:
# compile the model
model_5.compile(loss='binary_crossentropy',
                optimizer=Adam(),
                metrics=['accuracy'])

In [ ]:
## fit the model
history_5 = model_5.fit(train_data,
                        epochs=5,
                        steps_per_epoch=len(train_data),
                        validation_data=test_data,
                        validation_steps=len(test_data))

In [ ]:
model_5.summary()

In [ ]:
# plot loss curves
plot_loss_curves(history_5)

> **reducing overfitting with *data augmentation***.
>> *Opening our bag of tricks*

In [ ]:
# create ImageDataGenerator training instance with data augmentation
train_datagen_augmented = ImageDataGenerator(rescale=1/255.,
                                             rotation_range=0.2,
                                             shear_range=0.2,
                                             zoom_range=0.2,
                                             width_shift_range=0.2,
                                             height_shift_range=0.3,
                                             horizontal_flip=True)

# create ImageDataGenerator without data augmentation
train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

> 🤔🤔 **Question** What is *data augmentation*?

Data augmentation is the process of altering our training data, leading it to have more diversity and in turn allowing our model to learn more generalize (hopefully) patterns. Altring might mean adjusting the rotation of an image, flipping it, cropping it or something similar to it.

Let's write some code to visualize data augmentation

In [ ]:
# import data  and augment it from training directory
print("Augmented training data")
train_data_augmented = train_datagen_augmented.flow_from_directory(train_dir,
                                                                   target_size=(224, 224),
                                                                   batch_size=32,
                                                                   class_mode='binary',
                                                                   shuffle=False) # for demonstration purpose only

# create non augmented train data batches
print('Non-Augmented train data')
train_data = train_datagen.flow_from_directory(train_dir,
                                               target_size=(224, 224),
                                               batch_size=32,
                                               class_mode='binary',
                                               shuffle=False)

IMG_SIZE = (224, 224)
# create non augmented train data batches
print('Non-Augmented test data')
test_data = test_datagen.flow_from_directory(test_dir,
                                             target_size=IMG_SIZE,
                                             batch_size=32,
                                             class_mode='binary',
                                             shuffle=False)

> 🔑 **Note**: Data augmentation is usually only performed on the trainibg data. Using `ImageDataGenerator` built-in data augmentation parameter our images are left as they are in directories but are modified as they're loaded into the model.

Finally Let's visualize some augmented data!!!

In [ ]:
# get sample data batches
images, labels = next(train_data)
augmented_images, augmented_labels = next(train_data_augmented) # note: labels aren't augmented... only data(images)


In [ ]:
## show the original image and augmented image
import random
random_number = random.randint(0, 32) # our batch size is 32...
print(f'Showing image number: {random_number}')
plt.imshow(images[random_number])
plt.title(f'original image')
plt.axis(False)
plt.figure()
plt.imshow(augmented_images[random_number])
plt.title(f'original image')
plt.axis(False)

Now we have seen what augmented training data looks like, let's build a model and see how it learns on augmented data

In [ ]:
# create the same model 
model_6 = Sequential([
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(pool_size=3),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(1, activation='sigmoid')  
])

# compile the model
model_6.compile (loss='binary_crossentropy',
                 optimizer=Adam(),
                 metrics=['accuracy'])

# Fit the model
## fit the model
history_6 = model_6.fit(train_data_augmented,
                        epochs=5,
                        steps_per_epoch=len(train_data_augmented),
                        validation_data=test_data,
                        validation_steps=len(test_data))

In [ ]:
# check our model training curves
plot_loss_curves(history_6)

In [ ]:
# import data  and augment it from training directory
print("Augmented training data")
train_data_augmented_shuffled = train_datagen_augmented.flow_from_directory(train_dir,
                                                                   target_size=(224, 224),
                                                                   batch_size=32,
                                                                   class_mode='binary',
                                                                   shuffle=True) 


In [ ]:
# create the same model (augmented and shuffle the data)
model_7 = Sequential([
    Conv2D(10, 3, activation='relu', input_shape=(224, 224, 3)),
    MaxPool2D(pool_size=3),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Conv2D(10, 3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(1, activation='sigmoid')  
])

# compile the model
model_7.compile (loss='binary_crossentropy',
                 optimizer=Adam(),
                 metrics=['accuracy'])

# Fit the model
## fit the model
history_7 = model_7.fit(train_data_augmented_shuffled,
                        epochs=5,
                        steps_per_epoch=len(train_data_augmented_shuffled),
                        validation_data=test_data,
                        validation_steps=len(test_data))

In [ ]:
# plot loss curves
plot_loss_curves(history_7)

### 7. repeat until satisfied

Since we've already beaten our baseline, there are few things which we coud try to continue to improve our model:

* Increase the number of model layers (e.g. add more `Conv2D`/`MaxPool2D` layers)
* Increase the number of filters in each convolutional layer (e.g. from 10 to 32 or even 64)
* Train for longer (more epochs)
* Find an ideal learning rate
* Get more data (give the model more opportunities to learn)
* Use **transfer learning** to leaverage what another image model has learn and adjust it for our own use case.

### Make prediction with our trained model on our own custom data

In [ ]:
# Classes we're working with
print(class_names)

In [ ]:
!rm 03-steak.jpeg

In [ ]:
!curl -L https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-steak.jpeg -o 03-steak.jpeg

# from PIL import Image
# from IPython.display import display

# display(Image.open("03-steak.jpeg"))

In [ ]:
!file 03-steak.jpeg

In [ ]:
# View our example image

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

steak = mpimg.imread('03-steak.jpeg')
plt.imshow(steak)
plt.axis(False)

In [ ]:
# check the shape of image
steak.shape

> **Note**: When you train a neral network and you want to predict with it on your own custom data, it's important that your custom data shall be preprocessed into the same format as the data on which the model is trained upon.

In [ ]:
expanded_steak = tf.expand_dims(steak, axis=0).shape

In [ ]:
# create a function to import an image and resize it to be able to be use with our model
def load_and_prep_image(filename, img_shape=224):
    ''' 
    Reads the image from filename, turns it into a tensor and reshape it to 
    (img_shape, img_shape, color_channel).
    '''
    img = tf.io.read_file(filename)
    
    # decode the read file into tensor
    img = tf.image.decode_image(img)
    
    # resize the image
    img = tf.image.resize(img, size=[img_shape, img_shape])
    
    # rescale the image (get all values between 0 & 1)
    img = img/255.
    
    return img

In [ ]:
# load in and preprocess the image
steak = load_and_prep_image('03-steak.jpeg')
steak

In [ ]:
# this is pretty large image we cannot pass directly
pred = model_7.predict(tf.expand_dims(steak, axis=0))

Looks like our custom imahe is being put through our model, however, it currently outputs a prediction probability, wouldn't it be nice if we could visualize the image as well as the model's prediction?

In [ ]:
# we can index the predicted class by rounding the prediction probability and indexing it on class names
pred_class = class_names[int(tf.round(pred))]
pred_class

In [ ]:
# function to predict and plot
def pred_and_plot(model, filename, class_names=class_names):
    ''' 
    Imports and image located at filename, makes a prediction with model and plots the image with the
    predicted class as the title.
    '''
    # Import the target image abd prprocess it
    img = load_and_prep_image(filename)
    
    # make a prediction
    pred = model.predict(tf.expand_dims(img, axis=0))
    
    # get the predicted class
    pred_class = class_names[int(tf.round(pred))]
    
    # plot the image and predicted class
    plt.imshow(img)
    plt.title(f"Prediction: {pred_class}")
    plt.axis(False);

In [ ]:
# test our model on a custom image
pred_and_plot(model_7, '03-steak.jpeg')

Our model works! lets try it on another image... this time pizza 🍕

In [ ]:
!curl -L https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-pizza-dad.jpeg -o 03-pizza-dad.jpeg

In [ ]:
!file 03-pizza-dad.jpeg

In [ ]:
pred_and_plot(model_7, '03-pizza-dad.jpeg')

In [ ]:
!curl -L https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-hamburger.jpeg -o 03-hamburger.jpeg

In [ ]:
!file 03-hamburger.jpeg

In [ ]:
pred_and_plot(model_7, '03-hamburger.jpeg')